# kvroute · Phase 2: vLLM on a T4

Runs one vLLM worker on GPU 0, then a smoke check, the calibration experiments, a fitted cost model and a plot. About 25-35 minutes, most of it installing vLLM and downloading the model.

**Before running:** in the notebook sidebar set *Accelerator* = **GPU T4 x2** and turn **Internet on** (Kaggle asks for phone verification once). Optional: *Add Input* → search "COCO 2017" and attach a dataset that has `val2017/` images. Without it, synthetic images are used, which is fine for timing.

Run all cells top to bottom. At the end, download `phase2.zip` from the Output panel.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!python --version

In [ ]:
# ~5-10 minutes. vLLM brings its own PyTorch build.
!pip install -q "vllm==0.32.0" 2>&1 | tail -3
!python -c "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__, 'cuda', torch.version.cuda)"

In [ ]:
BRANCH = 'main'   # or a feature branch, e.g. 'phase-2-vllm'
%cd /kaggle/working
!rm -rf kvsim && git clone -q -b {BRANCH} https://github.com/mansidhamne/kvsim.git
%cd /kaggle/working/kvsim
!git log --oneline -1
!pip install -q -e ".[router]" 2>&1 | tail -2

In [ ]:
# Use attached COCO images if there are any (first val2017 folder found under /kaggle/input)
import glob
found = sorted(glob.glob('/kaggle/input/**/val2017', recursive=True))
IMAGES = found[0] if found else ''
print('images:', IMAGES or 'synthetic')

In [ ]:
# Smoke check + calibration. If the smoke check fails, read the printed reason.
# If vLLM fails to start because of the attention backend, add: --attention-backend TRITON_ATTN
img_flag = f'--images {IMAGES}' if IMAGES else ''
!python -m deploy.kaggle_run phase2 --out /kaggle/working/phase2 {img_flag}

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/phase2/calibration_vllm.png'))

In [ ]:
%cd /kaggle/working
!zip -qr phase2.zip phase2 && ls -lh phase2.zip

Download `phase2.zip` (Output panel, right side) and unzip it into `calibration/results/vllm_t4/` in the repo. If anything failed, the useful files are `phase2/smoke.json` and `phase2/logs/vllm_8001.log`.